In [1]:
import sys
from pathlib import Path
# sys.path.insert(0, str(Path(__file__).parent.parent))

from typing import Optional

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import shap

import nfl_data_py as nfl

import os
from sqlalchemy import (
    create_engine, Column, Integer, Float, String, Boolean,
    Date, DateTime, Text, ForeignKey, Index, UniqueConstraint, text,
inspect
)
from sqlalchemy.orm import DeclarativeBase, relationship, Session
from sqlalchemy.pool import StaticPool

from datetime import date

pd.set_option('display.max_columns', None)

In [2]:
current_path = os.getcwd()
DB_PATH = str(Path(current_path).parent.parent / "dynasty_scout.db")
DB_PATH

'/Users/sandeeptiwari/Desktop/dynasty-ai-engine/dynasty_scout.db'

# Player Name Matching
___
There is no internal key to match the college data from CFBD to the NFL data from nfl_data_py. So I will be writing code to use the player names to match across datasets. This code will normalize the player name-based keys and fuzzy match. This should deterministically join NFL to college data.

## Load Data

In [3]:
def get_engine(db_path: Path = DB_PATH, echo: bool = False):
    """
    Returns a SQLAlchemy engine. Uses StaticPool so the same connection
    is reused in single-threaded contexts (fine for local use).
    """
    return create_engine(
        f"sqlite:///{db_path}",
        connect_args={"check_same_thread": False},
        poolclass=StaticPool,
        echo=echo,
    )

def _load_nfl_stats(seasons: list[int]) -> pd.DataFrame:
        query = f"""
            SELECT s.*, p.position, p.nfl_team as current_team,
                   p.birth_date, p.years_exp, p.draft_round, p.draft_pick,
                   p.rookie_year, p.sleeper_id, p.name as player_name
            FROM nfl_season_stats s
            JOIN players p ON s.player_id = p.player_id
            WHERE s.season IN ({','.join(map(str, seasons))})
              AND s.season_type = 'REG'
              AND p.position IN ('QB', 'RB', 'WR', 'TE')
        """
        with engine.connect() as conn:
            return pd.read_sql(text(query), conn)

def _load_college_stats() -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text("SELECT * FROM college_season_stats"), conn)

def _load_combine_data() -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text("SELECT * FROM combine_measurements"), conn)

In [4]:
seasons = list(range(2015, 2025))
engine = get_engine()

In [7]:
players_df = _load_nfl_stats(seasons)
# team_df = _load_team_context(seasons)
college_df = _load_college_stats()
combine_df = _load_combine_data()

,id,player_id,season,season_type,team,games,completions,attempts,passing_yards,passing_tds,interceptions,passing_epa,completion_pct,yards_per_attempt,passer_rating,sacks,sack_yards,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_epa,yards_per_carry,targets,receptions,receiving_yards,receiving_tds,receiving_fumbles,receiving_epa,yards_per_reception,catch_rate,yards_per_target,air_yards_total,yards_after_catch,fantasy_points_ppr,fantasy_points_half,fantasy_points_std,fantasy_ppg_ppr,fantasy_ppg_half,snap_pct,target_share,air_yards_share,racr,wopr,tgt_per_game,position,current_team,birth_date,years_exp,draft_round,draft_pick,rookie_year,sleeper_id,player_name
